In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

# Generujemy TYLKO normalne transakcje — fraudów nie potrzebujemy!
N_NORMAL = 2000

normal = pd.DataFrame({
    'amount':        np.random.lognormal(5, 1, N_NORMAL).clip(5, 5000),
    'is_electronics': np.random.binomial(1, 0.3, N_NORMAL),
    'tx_per_minute':  np.random.poisson(3, N_NORMAL),
})

print(f"Dane treningowe: {len(normal)} normalnych transakcji")
print()
normal.describe().round(2)

Dane treningowe: 2000 normalnych transakcji



,amount,is_electronics,tx_per_minute
count,2000.00,2000.00,2000.00
mean,253.77,0.28,2.96
std,324.84,0.45,1.65
min,5.81,0.00,0.00
25%,79.63,0.00,2.00
50%,155.20,0.00,3.00
75%,293.82,1.00,4.00
max,5000.00,1.00,10.00


In [23]:
from sklearn.ensemble import IsolationForest
import pickle

features = ['amount', 'is_electronics', 'tx_per_minute']
X_train = normal[features]

# contamination: spodziewamy się ~5% anomalii w strumieniu
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.1,
    random_state=42
)
iso_forest.fit(X_train)

print("Model wytrenowany.")
print(f"Liczba drzew: {iso_forest.n_estimators}")
print(f"Contamination: {iso_forest.contamination}")

# Zapisz model
with open('fraud_model_if.pkl', 'wb') as f:
    pickle.dump(iso_forest, f)
print("\nZapisano do fraud_model_if.pkl")

Model wytrenowany.
Liczba drzew: 100
Contamination: 0.1

Zapisano do fraud_model_if.pkl


In [24]:
test_cases = pd.DataFrame([
    {'amount': 150.0,  'is_electronics': 0, 'tx_per_minute': 3,  'opis': 'normalna'},
    {'amount': 89.0,   'is_electronics': 0, 'tx_per_minute': 2,  'opis': 'normalna'},
    {'amount': 4800.0, 'is_electronics': 1, 'tx_per_minute': 12, 'opis': 'podejrzana'},
    {'amount': 3500.0, 'is_electronics': 1, 'tx_per_minute': 9,  'opis': 'podejrzana'},
    {'amount': 250.0,  'is_electronics': 1, 'tx_per_minute': 5,  'opis': 'graniczna'},
])

X_test = test_cases[features]
preds  = iso_forest.predict(X_test)          # +1 lub -1
scores = iso_forest.decision_function(X_test) # anomaly score

test_cases['wynik']  = preds
test_cases['score']  = scores.round(4)
test_cases['anomalia'] = test_cases['wynik'] == -1

print(test_cases[['opis', 'amount', 'is_electronics', 'tx_per_minute',
                   'wynik', 'score', 'anomalia']].to_string(index=False))

      opis  amount  is_electronics  tx_per_minute  wynik   score  anomalia
  normalna   150.0               0              3      1  0.1787     False
  normalna    89.0               0              2      1  0.1770     False
podejrzana  4800.0               1             12     -1 -0.2260      True
podejrzana  3500.0               1              9     -1 -0.2222      True
 graniczna   250.0               1              5      1  0.0314     False


In [25]:
# Załaduj stary model RF z Ćw. 2
with open('fraud_model.pkl', 'rb') as f:
    rf_model = pickle.load(f)

# Wygeneruj mieszane dane testowe (normalne + fraudy)
np.random.seed(0)
test_normal = pd.DataFrame({
    'amount':        np.random.lognormal(5, 1, 50).clip(5, 3000),
    'is_electronics': np.random.binomial(1, 0.3, 50),
    'tx_per_minute':  np.random.poisson(3, 50),
    'true_label': 0
})
test_fraud = pd.DataFrame({
    'amount':        np.random.uniform(2000, 9000, 10),
    'is_electronics': np.random.binomial(1, 0.7, 10),
    'tx_per_minute':  np.random.poisson(8, 10),
    'true_label': 1
})
test_df = pd.concat([test_normal, test_fraud], ignore_index=True)
X_cmp   = test_df[features]

# Predykcje
rf_pred = rf_model.predict(X_cmp)                  # 0 lub 1
if_pred = (iso_forest.predict(X_cmp) == -1).astype(int)  # 0 lub 1

from sklearn.metrics import precision_score, recall_score, f1_score

y_true = test_df['true_label']

print(f"{'Model':<20} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 55)
for name, pred in [("Random Forest", rf_pred), ("Isolation Forest", if_pred)]:
    p = precision_score(y_true, pred, zero_division=0)
    r = recall_score(y_true, pred, zero_division=0)
    f = f1_score(y_true, pred, zero_division=0)
    print(f"{name:<20} {p:>10.3f} {r:>10.3f} {f:>10.3f}")

print()
print("Uwaga: dane testowe są syntetyczne — IF może wypaść gorzej bo nie widział")
print("fraudów podczas treningu. W produkcji (bez etykiet) IF jest jedyną opcją.")

Model                 Precision     Recall         F1
-------------------------------------------------------
Random Forest             1.000      0.900      0.947
Isolation Forest          0.500      1.000      0.667

Uwaga: dane testowe są syntetyczne — IF może wypaść gorzej bo nie widział
fraudów podczas treningu. W produkcji (bez etykiet) IF jest jedyną opcją.


In [15]:
%%file fraud_api.py
from fastapi import FastAPI
from pydantic import BaseModel
import pickle
import numpy as np

app = FastAPI(title="Fraud Detection API — Isolation Forest")

model = pickle.load(open('fraud_model_if.pkl', 'rb'))

class Transaction(BaseModel):
    amount: float
    is_electronics: int
    tx_per_minute: int

@app.post("/score")
def score(tx: Transaction):
    X = np.array([[tx.amount, tx.is_electronics, tx.tx_per_minute]])
    prediction     = model.predict(X)[0]           # +1 lub -1
    anomaly_score  = model.decision_function(X)[0]  # ujemny = bardziej podejrzany

    # Normalizujemy score do przedziału [0, 1] — dla spójności z Ćw. 2
    # decision_function typowo zwraca wartości z zakresu [-0.5, 0.5]
    fraud_probability = float(np.clip(0.5 - anomaly_score, 0.0, 1.0))

    return {
        "is_fraud":          bool(prediction == -1),
        "fraud_probability": round(fraud_probability, 4),
        "model":             "isolation_forest",
    }

@app.get("/health")
def health():
    return {"status": "ok"}

Overwriting fraud_api.py


In [27]:
import requests, time

time.sleep(1)  # daj chwilę na restart serwera

cases = [
    {"amount": 150,  "is_electronics": 0, "tx_per_minute": 3,  "opis": "normalna"},
    {"amount": 4800, "is_electronics": 1, "tx_per_minute": 12, "opis": "podejrzana"},
    {"amount": 89,   "is_electronics": 0, "tx_per_minute": 2,  "opis": "normalna"},
    {"amount": 3200, "is_electronics": 1, "tx_per_minute": 8,  "opis": "podejrzana"},
]

for case in cases:
    payload = {k: v for k, v in case.items() if k != 'opis'}
    r = requests.post("http://localhost:8001/score", json=payload)
    result = r.json()
    print(f"[{case['opis']:10s}] amount={case['amount']:5} "
          f"→ fraud={result['is_fraud']}, prob={result['fraud_probability']:.3f}")

[normalna  ] amount=  150 → fraud=False, prob=0.321
[podejrzana] amount= 4800 → fraud=True, prob=0.726
[normalna  ] amount=   89 → fraud=False, prob=0.323
[podejrzana] amount= 3200 → fraud=True, prob=0.710


In [18]:
# 1. Jakie transakcje IF flaguje, a RF nie (i odwrotnie)?
#    OBSERWACJA:
#    Isolation Forest flaguje transakcje, ktore odbiegaja od wzorca normalnych
#    transakcji, nawet jesli nie wygladaja dokladnie jak fraudy z danych treningowych.
#    Dlatego IF moze oznaczac jako podejrzane np. transakcje o sredniej kwocie,
#    ale nietypowej kombinacji cech, np. wysoka aktywnosc, elektronika albo
#    nietypowa wartosc amount wzgledem normalnego rozkladu.
#
#    Random Forest czesciej flaguje transakcje podobne do fraudow, ktore widzial
#    podczas treningu, np. bardzo wysokie kwoty, elektronika i duzo transakcji.
#    RF moze natomiast nie wykryc nowego typu fraudu, jezeli taki wzorzec nie
#    wystepowal w danych treningowych.

# 2. Czy parametr contamination=0.05 ma wpływ na liczbę alertów?
#    Co by się stało gdybyś zmienił go na 0.01?
#    ODPOWIEDŹ:
#    Tak, contamination ma wplyw na liczbe alertow. Parametr contamination okresla,
#    jaki odsetek obserwacji model ma traktowac jako anomalie.
#
#    Przy contamination=0.05 model zaklada, ze okolo 5% obserwacji to anomalie.
#    Gdybysmy zmienili contamination na 0.01, model bylby bardziej konserwatywny
#    i oznaczalby mniej transakcji jako podejrzane. Liczba alertow by spadla,
#    ale moglibysmy tez przeoczyc czesc faktycznie podejrzanych transakcji.

# 3. Jaką zaletę ma IF w systemie produkcyjnym gdzie fraudy są nowe i nieznane?
#    ODPOWIEDŹ:
#    Glowna zaleta Isolation Forest jest taka, ze nie potrzebuje etykiet fraudow
#    do treningu. Model uczy sie tylko tego, jak wygladaja normalne transakcje,
#    a potem wykrywa obserwacje odbiegajace od normy.
#
#    Dzieki temu IF moze byc przydatny w produkcji, gdy pojawiaja sie nowe typy
#    fraudow, ktorych nie bylo jeszcze w danych historycznych. W takim przypadku
#    model nadzorowany, np. Random Forest, moze ich nie rozpoznac, bo nie widzial
#    podobnych przykladow podczas treningu.

In [10]:

import os
for f in ['fraud_model.pkl', 'fraud_model_if.pkl', 'fraud_api.py', 'ml_consumer.py']:
    exists = os.path.exists(f)
    print(f"{'✓' if exists else '✗'} {f}")

✓ fraud_model.pkl
✓ fraud_model_if.pkl
✓ fraud_api.py
✓ ml_consumer.py


In [28]:
import requests

r = requests.get("http://localhost:8001/model-info")
print(r.json())

{'type': 'isolation_forest', 'n_estimators': 100, 'contamination': 0.1}


In [30]:
import requests
print(requests.get("http://localhost:8001/model-info").json())

{'type': 'isolation_forest', 'n_estimators': 100, 'contamination': 0.1}


In [32]:
import requests
print(requests.get("http://localhost:8002/model-info").json())

{'type': 'random_forest', 'n_estimators': 100}
